# Challenge 3: Music Genre Classification

## Huấn luyện mô hình cơ bản (Baseline Model Training)

Các mô hình được huấn luyện trên tập dữ liệu gốc (chưa qua feature engineering), nhằm đánh giá hiệu năng cơ bản ban đầu.

### Khai báo thư viện

In [14]:
import pandas as pd
import numpy as np
import os, sys, random
from IPython import display

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# ==== Ignore warnings ====
import warnings
warnings.filterwarnings("ignore")

### Tham số thực nghiệm

In [4]:
params = {}

# ===== Thư mục thí nghiệm =====
params["exps_dir"]  = "../exps"
params["exp_name"]  = "music_genre_baseline"
params["save_dir"]  = f'{params["exps_dir"]}/{params["exp_name"]}'

# Đường dẫn dữ liệu đã preprocess (before FE)
params["data_path"] = f'{params["exps_dir"]}/data/train.xlsx'
params["test_path"] = f'{params["exps_dir"]}/data/test.xlsx'

params["k_fold"] = 10
params["random_state"] = 42

os.makedirs(params["save_dir"], exist_ok=True)

random.seed(params["random_state"])
np.random.seed(params["random_state"])
os.environ["PYTHONHASHSEED"] = str(params["random_state"])

print("Save directory:", params["save_dir"])
print("Train path    :", params["data_path"])
print("Test path     :", params["test_path"])

Save directory: ../exps/music_genre_baseline
Train path    : ../exps/data/train.xlsx
Test path     : ../exps/data/test.xlsx


### Nạp dữ liệu

In [5]:
df_train = pd.read_excel(params["data_path"])
df_test  = pd.read_excel(params["test_path"])

In [6]:
df_train.head()

,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature,Class
0,0.363636,0.295244,0.535438,0.8,0.806485,0,0.016729,0.379518,0.003935,0.096011,0.221358,0.652214,0.138741,0.75,9
1,0.666667,0.715946,0.746693,1.0,0.833220,1,0.069812,0.027309,0.046987,0.093970,0.371695,0.547814,0.129947,0.75,6
2,0.434343,0.564235,0.803763,0.6,0.819925,1,0.042252,0.000972,0.637550,0.277625,0.636081,0.692479,0.109016,0.75,10
3,0.111111,0.489994,0.307162,0.6,0.611251,1,0.009330,0.910643,0.021385,0.293950,0.497149,0.759476,0.201797,0.50,2
4,0.474747,0.543792,0.776730,0.5,0.844094,0,0.242895,0.183735,0.003935,0.203143,0.619492,0.309078,0.172046,0.75,5


In [7]:
df_test.head()

,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature
0,0.434343,0.679363,0.669600,0.1,0.795733,0,0.076783,0.076004,3.534040e-02,0.188858,0.636081,0.317930,0.135392,0.75
1,0.131313,0.431892,0.776730,0.1,0.786628,1,0.008686,0.389558,9.267068e-01,0.284767,0.522032,0.709016,0.192195,0.75
2,0.797980,0.641704,0.290141,0.1,0.711484,1,0.007292,0.875502,3.934743e-03,0.104173,0.286677,0.392884,0.157416,0.75
3,0.515152,0.452335,0.825789,0.6,0.856057,1,0.018445,0.000800,1.004017e-08,0.115396,0.689995,0.350227,0.150321,0.75
4,0.222222,0.725629,0.728672,0.0,0.812975,0,0.279357,0.147590,3.934743e-03,0.056423,0.812338,0.243355,0.080178,0.75


In [9]:
# Tách X, y
y = df_train["Class"].copy()
X = df_train.drop(columns=["Class"]).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

# Split có stratify (bắt buộc khi classification)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train X:", X_train.shape)
print("Valid X:", X_valid.shape)
print("Train y:", y_train.shape)
print("Valid y:", y_valid.shape)

X shape: (14396, 14)
y shape: (14396,)
Train X: (11516, 14)
Valid X: (2880, 14)
Train y: (11516,)
Valid y: (2880,)


In [11]:
# Dùng StratifiedKFold 
from sklearn.model_selection import StratifiedKFold

kfold = StratifiedKFold(
    n_splits=params["k_fold"],
    shuffle=True,
    random_state=params["random_state"]
)

print(f"Total rows in X_train: {len(X_train)}\n")

for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"---- Fold {fold} ----")
    print(f"Train size: {len(train_idx)}")
    print(f"Valid size: {len(valid_idx)}")
    print(f"Train idx sample: {train_idx[:10]}")
    print(f"Valid idx sample: {valid_idx[:10]}")
    print()

Total rows in X_train: 11516

---- Fold 0 ----
Train size: 10364
Valid size: 1152
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [ 25  60  73  89 100 105 123 126 138 140]

---- Fold 1 ----
Train size: 10364
Valid size: 1152
Train idx sample: [ 0  1  2  4  5  6  7  8  9 10]
Valid idx sample: [  3  27  46  64  77  83  86  92  93 107]

---- Fold 2 ----
Train size: 10364
Valid size: 1152
Train idx sample: [ 0  1  2  3  4  6  7  8  9 11]
Valid idx sample: [ 5 10 16 17 21 31 35 41 43 45]

---- Fold 3 ----
Train size: 10364
Valid size: 1152
Train idx sample: [ 1  2  3  4  5  6  7  8  9 10]
Valid idx sample: [ 0 13 24 38 40 50 55 56 69 70]

---- Fold 4 ----
Train size: 10364
Valid size: 1152
Train idx sample: [ 0  1  2  3  4  5  7  8  9 10]
Valid idx sample: [  6  14  29  30  63  67  68  90 110 119]

---- Fold 5 ----
Train size: 10364
Valid size: 1152
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [12 15 26 34 58 61 75 79 85 88]

---- Fold 6 ----
Train size: 10365
Valid s

### Lựa chọn mô hình mặc định

Các mô hình sẽ huấn luyện: Logistic Regression, Random Forest, Gradient Boosting, XGBoost và LightGBM.

In [18]:
models = [
    (
        "Logistic Regression",
        LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",
            n_jobs=-1,
            random_state=params["random_state"]
        )
    ),
    (
        "Random Forest",
        RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=params["random_state"],
            n_jobs=-1
        )
    ),
    (
        "Gradient Boosting",
        GradientBoostingClassifier(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=3,
            random_state=params["random_state"]
        )
    ),
    (
        "XGBoost",
        XGBClassifier(
            n_estimators=800,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softprob",   # phân loại multi-class
            eval_metric="mlogloss",
            random_state=params["random_state"],
            n_jobs=-1
        )
    ),
    (
        "LightGBM",
        LGBMClassifier(
            n_estimators=800,
            learning_rate=0.05,
            max_depth=-1,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multiclass",
            random_state=params["random_state"],
            n_jobs=-1,
            verbose=-1,           # tắt bớt log
            force_row_wise=True   
        )
    ),
]

### Huấn luyện và đánh giá từng mô hình

In [19]:
results = []
baseline_results = {}

for name, model in models:
    print(f"===== Model: {name} =====")

    baseline_results[name] = {
        "acc": [],
        "f1w": []
    }

    skf = StratifiedKFold(
        n_splits=params["k_fold"],
        shuffle=True,
        random_state=params["random_state"]
    )

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[valid_idx], y.iloc[valid_idx]

        # Train
        model.fit(X_tr, y_tr)

        # Predict
        y_pred = model.predict(X_val)

        # Metrics
        acc = accuracy_score(y_val, y_pred)
        f1w = f1_score(y_val, y_pred, average="weighted")

        baseline_results[name]["acc"].append(acc)
        baseline_results[name]["f1w"].append(f1w)

        print(f" Fold {fold}: ACC = {acc:.4f} | F1-weighted = {f1w:.4f}")

    mean_acc = np.mean(baseline_results[name]["acc"])
    std_acc  = np.std(baseline_results[name]["acc"])
    mean_f1w = np.mean(baseline_results[name]["f1w"])
    std_f1w  = np.std(baseline_results[name]["f1w"])

    print(f"--> Mean ACC: {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"--> Mean F1w: {mean_f1w:.4f} ± {std_f1w:.4f}\n")

    results.append([
        name,
        mean_acc,
        mean_f1w
    ])

===== Model: Logistic Regression =====
 Fold 0: ACC = 0.4979 | F1-weighted = 0.4691
 Fold 1: ACC = 0.4972 | F1-weighted = 0.4655
 Fold 2: ACC = 0.4910 | F1-weighted = 0.4627
 Fold 3: ACC = 0.4875 | F1-weighted = 0.4516
 Fold 4: ACC = 0.4833 | F1-weighted = 0.4554
 Fold 5: ACC = 0.4736 | F1-weighted = 0.4428
 Fold 6: ACC = 0.5003 | F1-weighted = 0.4700
 Fold 7: ACC = 0.4781 | F1-weighted = 0.4500
 Fold 8: ACC = 0.4871 | F1-weighted = 0.4558
 Fold 9: ACC = 0.4580 | F1-weighted = 0.4280
--> Mean ACC: 0.4854 ± 0.0123
--> Mean F1w: 0.4551 ± 0.0123

===== Model: Random Forest =====
 Fold 0: ACC = 0.5083 | F1-weighted = 0.4932
 Fold 1: ACC = 0.5007 | F1-weighted = 0.4887
 Fold 2: ACC = 0.5257 | F1-weighted = 0.5111
 Fold 3: ACC = 0.5062 | F1-weighted = 0.4899
 Fold 4: ACC = 0.5049 | F1-weighted = 0.4917
 Fold 5: ACC = 0.5174 | F1-weighted = 0.5028
 Fold 6: ACC = 0.5372 | F1-weighted = 0.5199
 Fold 7: ACC = 0.4997 | F1-weighted = 0.4844
 Fold 8: ACC = 0.5101 | F1-weighted = 0.4943
 Fold 9: ACC

**Nhận xét kết quả thực nghiệm các mô hình**

- Logistic Regression đạt độ chính xác trung bình khoảng 48.54% và F1-weighted 45.51%.
Hiệu suất thấp và dao động nhẹ giữa các fold -> mô hình tuyến tính không mô tả tốt các quan hệ phi tuyến giữa các đặc trưng âm thanh và thể loại nhạc.

- Random Forest cải thiện khoảng +3% so với Logistic Regression, đạt ACC 51.13% và F1-weighted 49.67% -> các mô hình cây quyết định có khả năng học được mối quan hệ phi tuyến trong dữ liệu âm nhạc tốt hơn mô hình tuyến tính.

- Gradient Boosting đạt hiệu suất cao nhất trong các mô hình thử nghiệm (Accuracy: 54.01%, F1-weighted: 51.85%) -> khai thác rất tốt quan hệ phi tuyến và tương tác giữa các đặc trưng âm thanh, mức dao động thấp (±1.1%) giữa các fold cũng cho thấy mô hình khá ổn định.

- XGBoost đạt độ chính xác 51.10%, tương đương Random Forest.

- LightGBM đạt ACC 50.65%, F1-weighted 49.75%, gần tương đương XGBoost và Random Forest.
Tốc độ huấn luyện nhanh và độ ổn định tốt, nhưng hiệu suất chưa vượt Gradient Boosting

Kết luận: **Gradient Boosting** là mô hình hoạt động tốt nhất trong giai đoạn trước FE với Accuracy ~54%, cao hơn các mô hình còn lại từ 2–6%.